In [1]:
import sys
sys.path.append('..')
from os.path import join, exists
from collections import OrderedDict
from pickle import load
from tqdm import tqdm

from search_state import DerivationTreeNode, Operation

In [2]:
######################
# Old representation #
######################
#
# architecture = OrderedDict(
#     {
#         "fn": routing_module,
#         "children": OrderedDict(
#             {
#                 "prerouting_fn": OrderedDict({"fn": im2col1k2s0p}),
#                 "inner_fn": OrderedDict(
#                     {
#                         "fn": computation_module,
#                         "children": OrderedDict(
#                             {
#                                 "computation_fn": linear512
#                             }
#                         ),
#                     }
#                 ),
#                 "postrouting_fn": OrderedDict({"fn": col2im}),
#             }
#         ),
#     }
# )
#

######################
# New representation #
######################
#
# [
#     DerivationTreeNode(id=1, level=network, operation=Operation(routing, nonterminal, ['prerouting_fn', 'module', 'postrouting_fn'])),
#     DerivationTreeNode(id=2, level=prerouting_fn, operation=Operation(im2col(1, 2, 0), terminal, [])),
#     DerivationTreeNode(id=3, level=module, operation=Operation(computation, nonterminal, ['computation_fn'])),
#     DerivationTreeNode(id=4, level=computation_fn, operation=Operation(linear512, terminal, [])),
#     DerivationTreeNode(id=5, level=postrouting_fn, operation=Operation(col2im, terminal, [])),
# ]

In [3]:
class Converter:
    def translate_name(self, name):
        if name in ["sequential", "routing", "branching(2)", "branching(4)", "branching(8)", "computation"]:
            name = name.replace("(", "").replace("2", "").replace("4", "").replace("8", "").replace(")", "")
            name += "_module"
        if "im2col" in name:
            # convert im2col(a, b, c) to im2colakbscp for any a, b and c
            # extract a, b and c
            a, b, c = map(lambda x: x.strip(), name.split("(")[1].split(")")[0].split(","))
            # insert a, b and c in the new name
            name = f"im2col{a}k{b}s{c}p"
        if "clone" in name:
            # convert clone(a) to clonea for any a
            a = name.split("(")[1].split(")")[0]
            name = f"clone_tensor{a}"
        if "group" in name:
            # convert group(a, b) to group_dimasbd for any a, b
            a, b = map(lambda x: x.strip(), name.split("(")[1].split(")")[0].split(","))
            name = f"group_dim{a}s{b}d"
        if "cat" in name:
            # convert cat(a, b) to cat_tensorsbdat for any a, b
            a, b = map(lambda x: x.strip(), name.split("(")[1].split(")")[0].split(","))
            name = f"cat_tensors{b}d{a}t"
        if "dot_product" in name:
            if "scaled" in name:
                # convert dot_product(scaled=True) to scaled_dot_product
                name = "scaled_dot_product"
            else:
                # convert dot_product to dot_product
                name = "dot_product"
        if "add" in name:
            # convert add to add
            name = "add_tensors"
        if "permute" in name:
            # if two commas
            if name.count(",") == 2:
                # convert permute(0, 2, 1) to permute21
                a, b, c = map(lambda x: x.strip(), name.split("(")[1].split(")")[0].split(","))
                name = f"permute{b}{c}"
            # if three commas
            elif name.count(",") == 3:
                # convert permute(0, 2, 3, 1) to permute231
                a, b, c, d = map(lambda x: x.strip(), name.split("(")[1].split(")")[0].split(","))
                name = f"permute{b}{c}{d}"
        if "linear" in name:
            # convert linear(a) to lineara for any a
            a = name.split("(")[1].split(")")[0]
            name = f"linear{a}"
        if "relu" in name:
            # convert relu to leakyrelu
            name = "leakyrelu"
        if "pos_enc" in name:
            # convert positional_encoding to learnable_positional_encoding
            name = "learnable_positional_encoding"
        return name

    def convert_to_old(self, root):
        """
        Convert any architecture from the new representation to the old representation
        In a recursive manner that can handle any depth of the architecture
        Input:
            root: DerivationTreeNode (with children)
        Output:
            OrderedDict
        """
        if root.operation.name == "sequential":
            return OrderedDict({
                "fn": self.translate_name(root.operation.name),
                "children": OrderedDict({
                    "first_fn": self.convert_to_old(root.children[0]),
                    "second_fn": self.convert_to_old(root.children[1]),
                }),
                "input_shape": root.input_params["shape"],
                "output_shape": root.output_params["shape"],
                "depth": root.depth,
                "node_type": "nonterminal",
                "node_id": root.id,
            })
        elif root.operation.name == "routing":
            return OrderedDict({
                "fn": self.translate_name(root.operation.name),
                "children": OrderedDict({
                    "prerouting_fn": self.convert_to_old(root.children[0]),
                    "inner_fn": self.convert_to_old(root.children[1]),
                    "postrouting_fn": self.convert_to_old(root.children[2]),
                }),
                "input_shape": root.input_params["shape"],
                "output_shape": root.output_params["shape"],
                "depth": root.depth,
                "node_type": "nonterminal",
                "node_id": root.id,
            })
        elif root.operation.name == "branching(2)":
            return OrderedDict({
                "fn": self.translate_name(root.operation.name),
                "children": OrderedDict({
                    "branching_fn": self.convert_to_old(root.children[0]),
                    "inner_fn": [
                        self.convert_to_old(root.children[1]),
                        self.convert_to_old(root.children[2]),
                    ],
                    "aggregation_fn": self.convert_to_old(root.children[3]),
                }),
                "input_shape": root.input_params["shape"],
                "output_shape": root.output_params["shape"],
                "depth": root.depth,
                "node_type": "nonterminal",
                "node_id": root.id,
            })
        elif root.operation.name == "branching(4)":
            return OrderedDict({
                "fn": self.translate_name(root.operation.name),
                "children": OrderedDict({
                    "branching_fn": self.convert_to_old(root.children[0]),
                    "inner_fn": [
                        self.convert_to_old(root.children[1]),
                        self.convert_to_old(root.children[1]),
                        self.convert_to_old(root.children[1]),
                        self.convert_to_old(root.children[1]),
                    ],
                    "aggregation_fn": self.convert_to_old(root.children[2]),
                }),
                "input_shape": root.input_params["shape"],
                "output_shape": root.output_params["shape"],
                "depth": root.depth,
                "node_type": "nonterminal",
                "node_id": root.id,
            })
        elif root.operation.name == "branching(8)":
            return OrderedDict({
                "fn": self.translate_name(root.operation.name),
                "children": OrderedDict({
                    "branching_fn": self.convert_to_old(root.children[0]),
                    "inner_fn": [
                        self.convert_to_old(root.children[1]),
                        self.convert_to_old(root.children[1]),
                        self.convert_to_old(root.children[1]),
                        self.convert_to_old(root.children[1]),
                        self.convert_to_old(root.children[1]),
                        self.convert_to_old(root.children[1]),
                        self.convert_to_old(root.children[1]),
                        self.convert_to_old(root.children[1]),
                    ],
                    "aggregation_fn": self.convert_to_old(root.children[2]),
                }),
                "input_shape": root.input_params["shape"],
                "output_shape": root.output_params["shape"],
                "depth": root.depth,
                "node_type": "nonterminal",
                "node_id": root.id,
            })
        elif root.operation.name == "computation":
            return OrderedDict({
                "fn": self.translate_name(root.operation.name),
                "children": OrderedDict({
                    "computation_fn": self.convert_to_old(root.children[0]),
                }),
                "input_shape": root.input_params["shape"],
                "output_shape": root.output_params["shape"],
                "depth": root.depth,
                "node_type": "nonterminal",
                "node_id": root.id,
            })
        elif root.is_leaf():
            return OrderedDict({
                "fn": self.translate_name(root.operation.name),
                "input_shape": root.input_params["shape"],
                "output_shape": root.output_params["shape"],
                "depth": root.depth,
                "node_type": "terminal",
                "node_id": root.id,
            })
        else:
            raise ValueError(f"Unknown level {root.level}")

    def convert_to_new(self, root):
        """
        Convert any architecture from the old representation to the new representation
        In a recursive manner that can handle any depth of the architecture
        Input:
            root: OrderedDict (with children)
        Output:
            DerivationTreeNode
        """
        raise NotImplementedError

In [4]:
converter = Converter()

# test it
assert converter.translate_name("sequential") == "sequential_module"
assert converter.translate_name("routing") == "routing_module"
assert converter.translate_name("branching(2)") == "branching_module"
assert converter.translate_name("branching(4)") == "branching_module"
assert converter.translate_name("branching(8)") == "branching_module"
assert converter.translate_name("computation") == "computation_module"
assert converter.translate_name("im2col(1,2,0)") == "im2col1k2s0p"
assert converter.translate_name("im2col(16,16,8)") == "im2col16k16s8p"
assert converter.translate_name("clone(2)") == "clone_tensor2"
assert converter.translate_name("clone(4)") == "clone_tensor4"
assert converter.translate_name("clone(8)") == "clone_tensor8"
assert converter.translate_name("group(1,2)") == "group_dim1s2d"
assert converter.translate_name("group(8,1)") == "group_dim8s1d"
assert converter.translate_name("cat(2,1)") == "cat_tensors1d2t"
assert converter.translate_name("cat(4,2)") == "cat_tensors2d4t"
assert converter.translate_name("cat(8,3)") == "cat_tensors3d8t"
assert converter.translate_name("dot_product(scaled=True)") == "scaled_dot_product"
assert converter.translate_name("add(2)") == "add_tensors"
assert converter.translate_name("add(4)") == "add_tensors"
assert converter.translate_name("add(8)") == "add_tensors"
assert converter.translate_name("permute(0,2,1)") == "permute21"
assert converter.translate_name("permute(0,2,3,1)") == "permute231"
assert converter.translate_name("linear(16)") == "linear16"
assert converter.translate_name("relu") == "leakyrelu"
assert converter.translate_name("pos_enc") == "learnable_positional_encoding"

In [5]:
architecture = [
    DerivationTreeNode(id=1, level="network",
        operation=Operation("routing", "nonterminal", ['prerouting_fn', 'module', 'postrouting_fn'], None, None, None, None),
        input_params={"shape": [1, 3, 32, 32]}
    ),
    DerivationTreeNode(id=2, level="prerouting_fn",
        operation=Operation("im2col(1, 2, 0)", "terminal", [], None, None, None, None),
        input_params={"shape": [1, 3, 32, 32]}
    ),
    DerivationTreeNode(id=3, level="module",
        operation=Operation("computation", "nonterminal", ['computation_fn'], None, None, None, None),
        input_params={"shape": [1, 256, 3]}
    ),
    DerivationTreeNode(id=4, level="computation_fn",
        operation=Operation("linear(512)", "terminal", [], None, None, None, None),
        input_params={"shape": [1, 256, 512]}
    ),
    DerivationTreeNode(id=5, level="postrouting_fn",
        operation=Operation("col2im", "terminal", [], None, None, None, None),
        input_params={"shape": [1, 512, 16, 16]}
    ),
]
# set output_params
architecture[0].output_params = {"shape": [1, 512, 16, 16]}
architecture[1].output_params = {"shape": [1, 256, 3]}
architecture[2].output_params = {"shape": [1, 256, 512]}
architecture[3].output_params = {"shape": [1, 256, 512]}
architecture[4].output_params = {"shape": [1, 512, 16, 16]}

# connect the children
architecture[0].children = [architecture[1], architecture[2], architecture[4]
]
architecture[2].children = [architecture[3]]
# convert the architecture
converter = Converter()
converted_architecture = converter.convert_to_old(architecture[0])
print(converted_architecture)

OrderedDict([('fn', 'routing_module'), ('children', OrderedDict([('prerouting_fn', OrderedDict([('fn', 'im2col1k2s0p'), ('input_shape', [1, 3, 32, 32]), ('output_shape', [1, 256, 3]), ('depth', 0), ('node_type', 'terminal'), ('node_id', 2)])), ('inner_fn', OrderedDict([('fn', 'computation_module'), ('children', OrderedDict([('computation_fn', OrderedDict([('fn', 'linear512'), ('input_shape', [1, 256, 512]), ('output_shape', [1, 256, 512]), ('depth', 0), ('node_type', 'terminal'), ('node_id', 4)]))])), ('input_shape', [1, 256, 3]), ('output_shape', [1, 256, 512]), ('depth', 0), ('node_type', 'nonterminal'), ('node_id', 3)])), ('postrouting_fn', OrderedDict([('fn', 'col2im'), ('input_shape', [1, 512, 16, 16]), ('output_shape', [1, 512, 16, 16]), ('depth', 0), ('node_type', 'terminal'), ('node_id', 5)]))])), ('input_shape', [1, 3, 32, 32]), ('output_shape', [1, 512, 16, 16]), ('depth', 0), ('node_type', 'nonterminal'), ('node_id', 1)])


In [6]:
# load in an architecture from a search results file
def load_results(path):
    full_path = join(path, "search_results.pkl")
    if not exists(full_path):
        raise FileNotFoundError(f"No checkpoint found at {full_path}")
    results = load(open(full_path, "rb"))
    print(f"Successfully loaded {results['iteration'] + 1} result iterations")
    return results

In [7]:
path = "../results/einspace/addnist/evolution/seed=0/backtrack=True/mode=iterative/time_limit=300/max_id_limit=10000/depth_limit=20/mem_limit=4096"
results = load_results(path)

Successfully loaded 187 result iterations


In [8]:
converter = Converter()

for i in tqdm(range(len(results["rewards"])), desc="Converting architectures"):
    architecture, reward, _, _ = results["rewards"][i]
    old_representation = converter.convert_to_old(architecture[0])

Converting architectures: 100%|██████████| 187/187 [00:00<00:00, 2678.91it/s]
